# Base API Tool
`baseApiTool.py` provides the shared HTTP GET utility for all MAF API tools.
To see this implemented, see `adviceApiTool.ipynb`.

## AG2 → MAF Migration
| AG2 Pattern | MAF Equivalent |
|---|---|
| `langchain.tools.BaseTool` subclass | Plain class with `@tool`-decorated `__call__` |
| `BaseAPIToolInput(BaseModel)` with `args_schema` | Type-annotated `__call__` parameters — MAF introspects these directly |
| `name`, `description`, `args_schema` class attrs | Function name and docstring on `__call__` |
| `_run(self, **kwargs)` | `__call__(self, ...)` with explicit typed parameters |
| `generate_llm_config(tool)` in `baseAgent.py` | Removed — MAF reads annotations automatically |

The core HTTP logic (`build_params`, `parse_response`, `requests.get`) is unchanged.
Only the framework wrapper changes.

## Imports


In [ ]:
from typing import Dict, Any, Optional
from agent_framework import tool
import requests

## BaseAPITool Class
In AG2, this class inherited from `langchain.tools.BaseTool` and required:
- A separate `BaseAPIToolInput(BaseModel)` class defining parameters via Pydantic `Field`
- `args_schema` pointing at that input class so AutoGen could serialize the schema
- `_run(**kwargs)` as the execution entry point (the leading underscore was a LangChain convention)

In MAF, the same contract is expressed directly on `__call__`:
- Parameters become typed function arguments with docstring descriptions
- The `@tool` decorator reads annotations and the docstring to build the LLM-facing schema
- No separate input model, no `args_schema`, no `_run`

The `build_params` and `parse_response` helpers are kept as plain methods —
they were never framework-specific, just HTTP plumbing.

Subclasses override `parse_response` to reshape the API response before it reaches the agent.
Everything else is inherited and runs the same as before.


In [ ]:
class BaseAPITool:
    """
    Shared HTTP GET base for MAF API tools.
    Subclasses override parse_response() to reshape the API response.
    """

    base_url: str = ""

    def build_params(self, **kwargs) -> Dict[str, Any]:
        """Strip None values and the base_url key before sending to the API."""
        return {k: v for k, v in kwargs.items() if k != "base_url" and v is not None}

    def parse_response(self, response: requests.Response) -> Any:
        """Default: return the full JSON payload. Subclasses narrow this down."""
        return response.json()

    @tool
    def __call__(self, base_url: Optional[str] = None, **kwargs) -> Any:
        """
        Call an HTTP GET API endpoint.

        Args:
            base_url: API endpoint URL. Uses class-level default if not provided.
        """
        url = base_url or self.base_url
        params = self.build_params(**kwargs)
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            return self.parse_response(response)
        except requests.exceptions.RequestException as e:
            return {"Error": str(e)}